

conda env: mmdet-sep2023

## Init

In [19]:
from pathlib import Path
import pyprojroot
dir_proj = pyprojroot.here()
print("Project directory:", dir_proj)

import pandas as pd
import numpy as np

import torch
import mmdet
import mmcv
import platform
print("Python:", platform.python_version())
# At least 1 gpu is needed for this to run reasonably quickly.
print("PyTorch version:", torch.__version__) # 1.13.1 for DyHead
print("GPU available:", torch.cuda.is_available())
print("GPU device count:", torch.cuda.device_count())
print("CUDA version:", torch.version.cuda) # 11.7 for DyHead
print("PyTorch CUDA archs:", torch.cuda.get_arch_list())
print("mmcv version:", mmcv.__version__) # 1.6.0 for DyHead
print("mmdet version:", mmdet.__version__) # tracking dev-3.x for mmyolo conda; 2.28.2 otherwise

Project directory: /home/ck432/projects/vaping-gpt
Python: 3.8.17
PyTorch version: 1.13.1
GPU available: True
GPU device count: 4
CUDA version: 11.7
PyTorch CUDA archs: ['sm_37', 'sm_50', 'sm_60', 'sm_61', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'compute_37']
mmcv version: 1.6.0
mmdet version: 2.28.2


In [6]:
# Copied from score-videos.ipynb in ecig-vaping project.
model_name = "dyhead"
timestr = "20230325-193432"

dir_models = dir_proj / "models"

model_config = dir_models / str(model_name + ".py")
print("Model file:", model_config)
print("Find model:", model_config.is_file())

file_checkpoint = dir_proj / ('models/' + model_name + "-" + timestr +'.pth')
print("Checkpoint file:", file_checkpoint)
print("Find checkpoint:", file_checkpoint.is_file())

Model file: /home/ck432/projects/vaping-gpt/models/dyhead.py
Find model: True
Checkpoint file: /home/ck432/projects/vaping-gpt/models/dyhead-20230325-193432.pth
Find checkpoint: True


In [7]:
from mmdet.apis import init_detector, inference_detector, show_result_pyplot
import mmcv

# build the model from a config file and a checkpoint file
model = init_detector(model_config, str(file_checkpoint), device='cuda:0')

/home/ck432/.conda/envs/mmdet-sep2023/lib/python3.8/site-packages/mmcv/cnn/bricks/hsigmoid.py:36: UserWarning: In MMCV v1.4.4, we modified the default value of args to align with PyTorch official. Previous Implementation: Hsigmoid(x) = min(max((x + 1) / 2, 0), 1). Current Implementation: Hsigmoid(x) = min(max((x + 3) / 6, 0), 1).
  warnings.warn(
/home/ck432/projects/ecig-vaping/external/mmdetection/mmdet/models/dense_heads/anchor_head.py:116: UserWarning: DeprecationWarning: `num_anchors` is deprecated, for consistency or also use `num_base_priors` instead
  warnings.warn('DeprecationWarning: `num_anchors` is deprecated, '


load checkpoint from local path: /home/ck432/projects/vaping-gpt/models/dyhead-20230325-193432.pth


In [8]:
# Copied from video.ipynb
video_dir = Path("videos/GPT4_themes")
videos = list(video_dir.glob('**/*.mp4'))
print("Found", len(videos), "videos to analyze")

Found 102 videos to analyze


## Sampled analysis

In [18]:
# Based on code in video.ipynb
def analyze_videos_mmcv(videos, max_frames = 4, verbose = False):
    results = {}
    for video in videos:
        video_name = video.stem
        if (verbose):
            print(f"\nAnalyzing {video_name}")
        # Import video
        frames = load_video_mmcv(str(video), max_frames = max_frames, verbose = verbose)
        # Analyze frames
        result = analyze_frames_mmcv(frames, verbose = verbose)
        # Save results
        results[video_name] = result
    return(results)
    
def load_video_mmcv(video_path, frame_sample = 100,
                  max_frames = None, verbose = False):
    video = mmcv.VideoReader(video_path)

    # obtain basic information
    if verbose:
        print("Total frames:", len(video))
        print(video.width, video.height, video.resolution, video.fps)
    
    total_frames = len(video)

    if max_frames is None:
        frame_slice = slice(0, total_frames, frame_sample)
    else:
        step_size = int(np.ceil(total_frames / max_frames))
        frame_slice = slice(0, total_frames, step_size)

    frames = video[frame_slice]

    if verbose:
        print("Extracted frames:", len(frames))
    return frames

def analyze_frames_mmcv(frames, verbose = False):
    results = []
    for frame_i in frames:
        result = inference_detector(model, frame_i)
        results.append(result)
    # Run dyhead model
    return results

In [19]:
results = analyze_videos_mmcv(videos, verbose = True)


Analyzing fabio_fashion_3
Total frames: 406
720 1280 (720, 1280) 25.0
Extracted frames: 4


/home/ck432/projects/ecig-vaping/external/mmdetection/mmdet/datasets/utils.py:66: UserWarning: "ImageToTensor" pipeline is replaced by "DefaultFormatBundle" for batch inference. It is recommended to manually replace it in the test data pipeline in your config file.
  warnings.warn(



Analyzing chamillioneyes_fashion_1
Total frames: 386
720 1280 (720, 1280) 30.0
Extracted frames: 4

Analyzing fabio_fashion_2
Total frames: 436
720 1280 (720, 1280) 30.0
Extracted frames: 4

Analyzing fabio_fashion_18
Total frames: 237
720 1280 (720, 1280) 30.0
Extracted frames: 4

Analyzing fabio_fashion_19
Total frames: 251
720 1280 (720, 1280) 30.0
Extracted frames: 4

Analyzing fabio_fashion_14
Total frames: 219
720 1280 (720, 1280) 30.0
Extracted frames: 4

Analyzing fabio_fashion_4
Total frames: 503
720 1280 (720, 1280) 30.0
Extracted frames: 4

Analyzing fabio_fashion_17
Total frames: 261
720 1280 (720, 1280) 30.0
Extracted frames: 4

Analyzing fabio_fashion_8
Total frames: 443
720 1280 (720, 1280) 30.0
Extracted frames: 4

Analyzing fabio_fashion_9
Total frames: 400
720 1280 (720, 1280) 30.0
Extracted frames: 4

Analyzing fabio_fashion_12
Total frames: 434
720 1280 (720, 1280) 30.0
Extracted frames: 4

Analyzing fabio_fashion_1
Total frames: 224
720 1280 (720, 1280) 30.0
Extra

In [20]:
len(results)

102

In [27]:
results['fabio_fashion_3']
results['fabio_ecigs_1'][1]

[array([], shape=(0, 5), dtype=float32),
 array([], shape=(0, 5), dtype=float32),
 array([], shape=(0, 5), dtype=float32),
 array([], shape=(0, 5), dtype=float32),
 array([], shape=(0, 5), dtype=float32),
 array([], shape=(0, 5), dtype=float32),
 array([[7.7902830e-01, 2.4958495e+02, 1.2807602e+02, 5.8565320e+02,
         6.0633369e-02],
        [0.0000000e+00, 4.4516656e+02, 1.6996367e+02, 6.3167938e+02,
         5.6264609e-02],
        [3.2896750e+02, 8.9905157e+02, 4.5574078e+02, 1.2591453e+03,
         5.5885814e-02],
        [3.5267091e-01, 3.8996368e+01, 1.2992154e+02, 5.9218970e+02,
         5.0376859e-02],
        [4.5845648e+02, 4.4411649e+02, 7.1654413e+02, 7.5474823e+02,
         4.8534099e-02],
        [3.2571033e+02, 8.3895825e+02, 5.5125507e+02, 1.2714786e+03,
         4.4728354e-02],
        [0.0000000e+00, 3.5391766e+02, 1.6205618e+02, 6.0842877e+02,
         4.2382803e-02],
        [0.0000000e+00, 3.0163275e+02, 1.8201967e+02, 6.8926947e+02,
         3.6258545e-02]], d

In [24]:
results.keys()

dict_keys(['fabio_fashion_3', 'chamillioneyes_fashion_1', 'fabio_fashion_2', 'fabio_fashion_18', 'fabio_fashion_19', 'fabio_fashion_14', 'fabio_fashion_4', 'fabio_fashion_17', 'fabio_fashion_8', 'fabio_fashion_9', 'fabio_fashion_12', 'fabio_fashion_1', 'fabio_fashion_16', 'fabio_fashion_11', 'fabio_fashion_7', 'fabio_fashion_15', 'fabio_fashion_6', 'fabio_fashion_13', 'fabio_fashion_10_andpod', 'fabio_fashion_5', 'fabio_health_3', 'drewdirps_health_2', 'fabio_health_2_and_pod', 'arabella_health_1', 'chamil_health_vape', 'chamillioneyes_health_1', 'edripps_health_vape', 'drewdirps_health_4', 'drewdirps_health_1', 'vapes_aby_health_1_andvape', 'vapes_aby_health_2_andvape', 'alex_health_vape', 'drew_health_vape', 'chamillioneyes_health_2', 'vapes_aby_health_vape', 'fabio_health_1', 'drewdirps_health_3', 'alex_health_2', 'arabella_health_2', 'alex_health_1', 'drewdirps_ecigs_3', 'chamillioneyes_ecgis_1', 'drewdirps_ecigs_4', 'edripss_ecgis_2', 'chamillioneyes_ecigs_2', 'calitrickzz_ecigs_3

In [34]:
# from compile-video-results.ipynb

import pickle

def analyze_video_detections(video_preds, video_name = "undefined"):
    # Each result contains the equivalent of the json - one element per frame, and each frame has a list of object detections for each class.

    n_frames =  len(video_preds)
    print("Frames:", n_frames)
    
    # Current class order (9):
    classes = ('box', 'e-cigarette brand name', 'e-juice', 'e-juice flavor', 'mod', 'pod', 'smoke cloud', 'synthetic nicotine label', 'warning label nicotine')
    
    # Define the columns that we want to track
    df_columns = (
                'video',
                'frame',
    ) + classes
    
    # Create a dataframe to store the frame counts for each object.
    video_df = \
        pd.DataFrame(0.,
                     index = np.arange(n_frames),
                     columns = classes)

    # Loop over each frame and extract the highest predicted prob for each class.
    for frame_i, frame in enumerate(video_preds):
                
        # These are the object detections that we want to count
        #pred_types = ('mod',
        #              'pod',
        #              'e-juice',
        #              'box',
        #              'smoke cloud')
        pred_types = classes
            
        # Loop over each prediction type that we care about and keep the highest probability for that class on this frame.
        for pred_type_i in pred_types:
            
            df_pred_col_i = list(video_df.columns).index(pred_type_i)
            
            # Find the location of this object type in the model result object.
            pred_index = classes.index(pred_type_i)

            # Count the number of predictions for this object that exceed the probability threshold for detection.
            # Extract the probability prediction for any detected objects
            frame_probs = [pred_i[4] for pred_i in frame[pred_index]]

            if len(frame_probs) > 0:
                max_prob = np.max(frame_probs)
            else:
                max_prob = 0.

            video_df.loc[frame_i, pred_type_i] = max_prob
            
    video_df['video'] = video_name
    video_df['frame'] = np.arange(n_frames)
    return(video_df)

In [37]:
video_dfs = []
for video_i, video_name in enumerate(results):
    video_result = results[video_name]
    print(f"{video_i}: {video_name}")
    video_df = analyze_video_detections(video_result, video_name)
    video_dfs.append(video_df)
        
# Combine all predictions into a single dataframe.
pred_df = pd.concat(video_dfs)

pred_df.to_parquet(dir_proj / "data/video-theme-frame-analysis.parquet")

0: fabio_fashion_3
Frames: 4
1: chamillioneyes_fashion_1
Frames: 4
2: fabio_fashion_2
Frames: 4
3: fabio_fashion_18
Frames: 4
4: fabio_fashion_19
Frames: 4
5: fabio_fashion_14
Frames: 4
6: fabio_fashion_4
Frames: 4
7: fabio_fashion_17
Frames: 4
8: fabio_fashion_8
Frames: 4
9: fabio_fashion_9
Frames: 4
10: fabio_fashion_12
Frames: 4
11: fabio_fashion_1
Frames: 4
12: fabio_fashion_16
Frames: 4
13: fabio_fashion_11
Frames: 4
14: fabio_fashion_7
Frames: 4
15: fabio_fashion_15
Frames: 4
16: fabio_fashion_6
Frames: 4
17: fabio_fashion_13
Frames: 4
18: fabio_fashion_10_andpod
Frames: 4
19: fabio_fashion_5
Frames: 4
20: fabio_health_3
Frames: 4
21: drewdirps_health_2
Frames: 4
22: fabio_health_2_and_pod
Frames: 4
23: arabella_health_1
Frames: 4
24: chamil_health_vape
Frames: 4
25: chamillioneyes_health_1
Frames: 4
26: edripps_health_vape
Frames: 4
27: drewdirps_health_4
Frames: 4
28: drewdirps_health_1
Frames: 4
29: vapes_aby_health_1_andvape
Frames: 4
30: vapes_aby_health_2_andvape
Frames: 4


In [38]:
pred_df

,box,e-cigarette brand name,e-juice,e-juice flavor,mod,pod,smoke cloud,synthetic nicotine label,warning label nicotine,video,frame
0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,fabio_fashion_3,0
1,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,fabio_fashion_3,1
2,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,fabio_fashion_3,2
3,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,fabio_fashion_3,3
0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.047046,0.0,0.000000,chamillioneyes_fashion_1,0
...,...,...,...,...,...,...,...,...,...,...,...
3,0.0,0.126527,0.049926,0.0,0.322694,0.313811,0.064886,0.0,0.043899,calitrickzz_mj_7,3
0,0.0,0.000000,0.000000,0.0,0.619724,0.050658,0.063027,0.0,0.065033,calitrickzz_mj_15,0
1,0.0,0.000000,0.000000,0.0,0.591773,0.056056,0.073923,0.0,0.050353,calitrickzz_mj_15,1
2,0.0,0.000000,0.000000,0.0,0.498510,0.090947,0.147942,0.0,0.000000,calitrickzz_mj_15,2


In [40]:
pred_df.to_excel("tables/dyhead-frame-predictions.xlsx")

## Full frame analysis

Pulling code from score-videos.ipynb in the ecig-vaping project.

In [9]:
videos

[PosixPath('videos/GPT4_themes/fashion/fabio_fashion_3.mp4'),
 PosixPath('videos/GPT4_themes/fashion/chamillioneyes_fashion_1.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_2.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_18.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_19.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_14.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_4.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_17.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_8.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_9.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_12.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_1.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_16.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_11.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_7.mp4'),
 PosixPath('videos/GPT4_themes/fashion/fabio_fashion_1

In [12]:
# Create video_df
video_df = pd.DataFrame({"video_path": videos})

In [14]:
video_df.describe()

,video_path
count,102
unique,102
top,videos/GPT4_themes/fashion/fabio_fashion_3.mp4
freq,1


In [39]:
# Modified from video-evaluation.ipynb
import os

def predict_video(video_name, video_dir,
                  dir_output = pyprojroot.here() / "data/video-frame-analysis",
                  box_threshold = 0.4,
                  overwrite = False,
                  verbose = False):

    print("Analyzing", video_name)

    video_file = video_dir / video_name
    
    if not os.path.exists(video_file):
        print("Could not find video file")
        raise Exception
    
    # This will contain the .mp4 file extension.
    path_out = dir_output / str(Path(video_file).stem + "-scored.mp4")
    if verbose:
        print("Output path:", path_out)
    
    if path_out.is_file() and not overwrite:
        print("Skipping file - output already exists.")
        return
    
    # This will create an output mp4 and an output pkl (to be analyzed).
    # Default threshold is 0.3
    # Previously notebooks/video_demo-ck.py in the ecig-vaping project.
    #!python analyze-single-video.py "{video_file}" {path_model} {file_checkpoint} --out "{path_out}" --save_result --score-thr {box_threshold}
    # %run analyze-single-video.py "{video_file}" {path_model} {file_checkpoint} --out "{path_out}" --save_result --score-thr {box_threshold}
    os.system(f'python analyze-single-video.py "{video_file}" {model_config} {file_checkpoint} --out "{path_out}" --save_result --score-thr {box_threshold}')

    """!python analyze-single-video.py "{video_file}" \
        {path_model} \
        {file_checkpoint} \
        --out "{path_out}" \
        --save_result \
        --score-thr {box_threshold}"""
    

In [ ]:
%%time
# Takes 14.3 hours
print(f"Analyzing {video_df.shape[0]:,} videos.")
for index, row in video_df.iterrows():
    print(f"Video {index}")
    video_path = Path(row['video_path'])
    video_name = video_path.name
    video_dir = video_path.parent
    # TODO: check if video has already been analyzed.
    # Second argument is the probability threshold for showing a bounding box.
    predict_video(video_name, video_dir, #dir_output = dir_output,
                  box_threshold = 0.4, verbose = True)

## Analyze scored videos

In [45]:
# Next, translate hte code from compile-video-result.ipynb to analyze the pickle dataframes for detected objects.
# Extract all videos that have been scored
dir_output = pyprojroot.here() / "data/video-frame-analysis"
scored_videos = list(dir_output.glob("*.pkl"))
print("Scored videos found:", len(scored_videos))

Scored videos found: 102


In [46]:
import pickle

def analyze_video_pkl(video_pkl):
    video_preds = pickle.load(open(video_pkl, 'rb'))
    video_name = video_pkl.stem
    # Each pickle contains the equivalent of the json - one element per frame, and each frame has a list of object detections for each class.
    
    n_frames =  len(video_preds)
    print("Frames:", n_frames)
    
    # Current class order (9):
    classes = ('box', 'e-cigarette brand name', 'e-juice', 'e-juice flavor', 'mod', 'pod', 'smoke cloud', 'synthetic nicotine label', 'warning label nicotine')
    
    # Define the columns that we want to track
    df_columns = (
                'video',
                'frame',
    ) + classes
    
    # Create a dataframe to store the frame counts for each object.
    #video_df = \
    #    pd.DataFrame(0,
    #                 index = np.arange(n_frames),
    #                 columns = df_columns)
    
    video_df = \
        pd.DataFrame(0.,
                     index = np.arange(n_frames),
                     columns = classes)

    #video_df['frame'] = np.arange(n_frames)
    #video_df['video'] = video_name

    
    # Loop over each frame and extract the highest predicted prob for each class.
    for frame_i, frame in enumerate(video_preds):
                
        # These are the object detections that we want to count
        #pred_types = ('mod',
        #              'pod',
        #              'e-juice',
        #              'box',
        #              'smoke cloud')
        pred_types = classes
            
        # Loop over each prediction type that we care about and keep the highest probability for that class on this frame.
        for pred_type_i in pred_types:
            
            df_pred_col_i = list(video_df.columns).index(pred_type_i)
            
            # Find the location of this object type in the model result object.
            pred_index = classes.index(pred_type_i)

            # Count the number of predictions for this object that exceed the probability threshold for detection.
            # Extract the probability prediction for any detected objects
            frame_probs = [pred_i[4] for pred_i in frame[pred_index]]

            if len(frame_probs) > 0:
                max_prob = np.max(frame_probs)
            else:
                max_prob = 0.
            #print(f"Updating: {frame_i}, {pred_type_i}, {max_prob}")
            #print(len(video_df[pred_type_i]))
            #print(video_df.loc[frame_i])
            #print(video_df.loc[frame_i])

            #video_df.loc[int(frame_i), pred_type_i] = max_prob
            video_df.loc[frame_i, pred_type_i] = max_prob
            #video_df.loc[frame_i, pred_type_i] = max_prob


            #video_df.at[frame_i, df_pred_col_i] = max_prob
    video_df['video'] = video_name
    video_df['frame'] = np.arange(n_frames)
    return(video_df)

In [47]:
%%time

video_dfs = []
for video_i, video_pkl in enumerate(scored_videos):
    print(f"{video_i}: {video_pkl.stem}")
    output_file = dir_output / (video_pkl.stem + ".parquet")
    print(output_file)
    if not output_file.exists():
        video_df = analyze_video_pkl(video_pkl)
        video_df.to_parquet(output_file)
    else:
        video_df = pd.read_parquet(output_file)
    video_dfs.append(video_df)
        
# Combine all predictions into a single dataframe.
pred_df = pd.concat(video_dfs)

pred_df.to_parquet(dir_proj / "data/unlabeled-video-frames.parquet")

0: fabio_health_2_and_pod-scored
/home/ck432/projects/vaping-gpt/data/video-frame-analysis/fabio_health_2_and_pod-scored.parquet
Frames: 1802
1: fabio_fashion_17-scored
/home/ck432/projects/vaping-gpt/data/video-frame-analysis/fabio_fashion_17-scored.parquet
Frames: 261
2: arabella_tech_4-scored
/home/ck432/projects/vaping-gpt/data/video-frame-analysis/arabella_tech_4-scored.parquet
Frames: 1609
3: fabio_fashion_8-scored
/home/ck432/projects/vaping-gpt/data/video-frame-analysis/fabio_fashion_8-scored.parquet
Frames: 443
4: zanxlyfe_tech_3-scored
/home/ck432/projects/vaping-gpt/data/video-frame-analysis/zanxlyfe_tech_3-scored.parquet
Frames: 1604
5: arabella_tech_2-scored
/home/ck432/projects/vaping-gpt/data/video-frame-analysis/arabella_tech_2-scored.parquet
Frames: 1719
6: alex_health_2-scored
/home/ck432/projects/vaping-gpt/data/video-frame-analysis/alex_health_2-scored.parquet
Frames: 431
7: arabella_tech_9-scored
/home/ck432/projects/vaping-gpt/data/video-frame-analysis/arabella_te

These results are then analyzed in `analyze-dyhead-predictions.Rmd`